In [1]:
import pandas as pd

df = pd.read_csv("data_for_embeddings_part_3.csv")

df = df['text']

print(df.shape)

df = df.to_frame()
print(df.shape)

df = df.head(100)
print(df.shape)


(10000,)
(10000, 1)
(100, 1)


In [6]:
import faiss
import numpy as np
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, pipeline
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings

# Load the DistilBERT tokenizer and model for question answering
tokenizer = DistilBertTokenizer.from_pretrained("fine_tuned_bert")
model = DistilBertForSequenceClassification.from_pretrained("fine_tuned_bert")

# Generate embeddings using DistilBERT
def generate_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.logits.squeeze()  # Use the logits as the embedding (or choose the last hidden state)
    return embeddings.numpy()  # Convert tensor to numpy array

# Create a FAISS index for the embeddings
dimension = model.config.hidden_size  # 768 for DistilBERT
index = faiss.IndexFlatL2(dimension)

# Assuming df['text'] contains the text data
texts = df['text'].tolist()

# Generate embeddings and ensure they are in a 2D array
embeddings = np.array([generate_embeddings(text) for text in texts])

# Ensure the embeddings are in the correct shape (num_embeddings, embedding_dimension)
embeddings = embeddings.reshape(-1, dimension)

# Add embeddings to the FAISS index
index.add(embeddings)

# Create a docstore for storing metadata (text documents)
docstore = {i: Document(page_content=text) for i, text in enumerate(texts)}

# Create a map from index to docstore IDs
index_to_docstore_id = {i: str(i) for i in range(len(texts))}

# Define the embedding function to pass to FAISS
def embedding_function(texts: list):
    return np.array([generate_embeddings(text) for text in texts])

# Create the FAISS vector store with the embedding function
faiss_store = FAISS(
    index=index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id,
    embedding_function=embedding_function
)

# For HuggingFace language models like T5, BERT, etc., use HuggingFacePipeline
# Initialize the HuggingFace pipeline for question answering or summarization, etc.
qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=qa_pipeline,
    retriever=faiss_store.as_retriever()
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 7.2 MB/s eta 0:00:00
